In [1]:
import sys
import numpy as np
import scipy
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy
from sympy import *

In [7]:
from sympy.abc import y

import numba

f = lambdify(y, sin(y), 'numpy')
g = numba.jit(f)

g(1)


0.8414709848078965

In [12]:
import sympy as sp
from numba import njit


import numpy as np
import sympy as sp
from numba import njit

# ------------------------------
# 1️⃣ Equations as strings
# ------------------------------
eq_strings = ["a*x - y", "y - b*x"]
var_names = ["x", "y"]
par_names = ["a", "b"]

# ------------------------------
# 2️⃣ Sympify
# ------------------------------
variables = sp.symbols(var_names)
parameters = sp.symbols(par_names)

exprs = [sp.sympify(s) for s in eq_strings]

# ------------------------------
# 3️⃣ Build Numba-compatible ODE
# ------------------------------
def make_numba_ODE(exprs, variables, parameters):
    n_eq = len(exprs)
    var_names = [str(v) for v in variables]
    par_names = [str(p) for p in parameters]

    code = "def ODE(v, p):\n"
    # map v -> variables
    for i, var in enumerate(var_names):
        code += f"    {var} = v[{i}]\n"
    # map p -> parameters
    for i, par in enumerate(par_names):
        code += f"    {par} = p[{i}]\n"
    # allocate output
    code += f"    out = np.empty({n_eq}, dtype=np.float64)\n"
    # assign expressions
    for i, expr in enumerate(exprs):
        code += f"    out[{i}] = {sp.ccode(expr)}\n"
    code += "    return out\n"

    ns = {"np": np}
    exec(code, ns)
    return njit(ns["ODE"])

ODE = make_numba_ODE(exprs, variables, parameters)

# ------------------------------
# 4️⃣ Test
# ------------------------------
v = np.array([1.0, 2.0])  # x=1, y=2
p = np.array([3.0, 4.0])  # a=3, b=4
print("ODE output:", ODE(v, p))



ODE output: [ 1. -2.]


In [18]:
import sympy as sp
import numpy as np
from numba import njit

# --- Step 1: Define equations as strings ---
eq_strings = ["x - y", "y - x"]  # updated equations
var_names = ["x", "y"]
par_names = []  # only one parameter now

# --- Step 2: Symbols ---
variables = sp.symbols(var_names)
parameters = sp.symbols(par_names)

# --- Step 3: Sympify expressions ---
exprs = [sp.sympify(s) for s in eq_strings]

# --- Step 4: Lambdify expressions ---
#lambdas = [sp.lambdify(variables + parameters, expr, "numpy") for expr in exprs]

# --- Step 5: Build ODE function ---
def make_numba_ODE(exprs, variables, parameters):
    n_eq = len(exprs)
    var_names = [str(v) for v in variables]
    par_names = [str(p) for p in parameters]

    code = "def ODE(v, p):\n"
    # map v -> variables
    for i, var in enumerate(var_names):
        code += f"    {var} = v[{i}]\n"
    # map p -> parameters
    for i, par in enumerate(par_names):
        code += f"    {par} = p[{i}]\n"
    # allocate output
    code += f"    out = np.empty({n_eq}, dtype=np.float64)\n"
    # assign expressions
    for i, expr in enumerate(exprs):
        code += f"    out[{i}] = {sp.ccode(expr)}\n"
    code += "    return out\n"

    ns = {"np": np}
    exec(code, ns)
    return njit(ns["ODE"])

ODE = make_numba_ODE(exprs, variables, parameters)

# --- Step 6: RK2 integrator ---
@njit
def rk2(f, t_eval, y0, params):
    y = np.zeros((len(t_eval), len(y0)))
    y[0] = y0
    for i in range(len(t_eval) - 1):
        h = t_eval[i+1] - t_eval[i]
        k1 = f(y[i], params)
        k2 = f(y[i] + 0.5*h*k1, params)
        y[i+1] = y[i] + 0.5*h*(k1 + k2)
    return y

# --- Step 7: Test ---
t = np.linspace(0, 2, 5)
y0 = np.array([1.0, 0.0])
params = np.array([])  # a = 1

sol = rk2(ODE, t, y0, params)
print("RK2 solution:\n", sol)


RK2 solution:
 [[  1.           0.        ]
 [  1.625       -0.625     ]
 [  3.03125     -2.03125   ]
 [  6.1953125   -5.1953125 ]
 [ 13.31445312 -12.31445312]]
